# HARI 3 — Menyimpan Konfigurasi LLM/RAG, requirements.txt, dan Dokumen Kontrak
**Lanjutan notebook Hari 2 Bagian 2.** Salin semua sel di notebook ini, lalu tempel di **bawah sel terakhir** notebook Hari 2.

Cara menyalin di Colab: klik sel pertama, tahan **Shift** lalu klik sel terakhir, tekan **Ctrl+C**; buka notebook Hari 2, klik sel paling bawah, tekan **Ctrl+V**.

**Sel Hari 2 yang harus sudah dijalankan (dalam sesi yang sama):**

| Untuk | Wajib dijalankan |
|---|---|
| Sel 1 (RAG) | 2.1, 2.3, 2.4 (2.6 dianjurkan agar `BATAS_JARAK` pilihan tim terbawa) |
| Sel 2 (LLM classification) | 2.1, 2.2, 2.8 |

Tim RAG boleh melewati Sel 2, dan tim classification dengan LLM boleh melewati Sel 1. Sel 4 otomatis hanya menulis kontrak untuk konfigurasi yang file-nya ada.

⚠️ Jika sesi Colab terputus, semua variabel hilang. Jalankan ulang sel wajib Hari 2 terlebih dahulu.

## RAG vs LLM Classification: apa yang disimpan?

Pada Bagian 2, LLM **tidak dilatih**, sehingga tidak ada file model `.pkl`. LLM tetap berada di server Groq dan dipanggil lewat API. Yang disimpan adalah **data dan pengaturan** agar backend menjalankan alur yang sama persis dengan notebook.

| Aspek | RAG | LLM Classification |
|---|---|---|
| Pertanyaan yang dijawab | "Apa jawaban pertanyaan ini **menurut dokumen**?" | "Kode ini termasuk **kategori** apa?" |
| Input saat dipakai | Teks pertanyaan | Teks/kode dari pengguna |
| Output | Kalimat jawaban (atau "tidak ditemukan") | Satu kategori dari daftar |
| Konfigurasi final | Batas jarak, jumlah dokumen (k), template prompt | Zero-shot atau few-shot (akurasi tertinggi) |
| Artefak | Folder `data/vectordb` + `konfigurasi_rag.json` | `konfigurasi_klasifikasi.json` |
| Metrik | Recall@k, MRR, % jawaban benar | Akurasi evaluation set |

- **Kenapa JSON, bukan .pkl?** Isinya hanya teks dan angka pengaturan, bukan model. JSON bisa dibuka dan dibaca langsung oleh backend dengan bahasa apa pun (Python, PHP, JavaScript).
- **RAG menyimpan vector database**, karena tahap Retrieve membutuhkan dokumen yang sudah diubah menjadi embedding. Folder ini sudah tersimpan sejak langkah 2.4 Hari 2.
- **Template prompt ikut disimpan**, karena jawaban LLM sangat bergantung pada kalimat prompt. Jika backend mengubah prompt, hasilnya bisa berbeda dari hasil evaluasi.

## Sel 1 – RAG: Menyimpan Konfigurasi dan Simulasi Tanya Jawab
Seluruh pengaturan RAG (lokasi database, model embedding, batas jarak, model LLM, template prompt) disimpan ke `models/konfigurasi_rag.json`. Setelah disimpan, file dibuka kembali lalu dipakai untuk menjawab 3 pertanyaan, sehingga terbukti bahwa konfigurasi tersebut cukup untuk menjalankan RAG.

In [ ]:
# SEL 1 - RAG: simpan konfigurasi RAG + cek vector database
import os
import json
import time

try:
    BATAS_JARAK               # dari langkah 2.6 Hari 2
except NameError:
    BATAS_JARAK = 0.8         # jika langkah 2.6 belum dijalankan

# Vector database SUDAH tersimpan di data/vectordb sejak langkah 2.4 Hari 2.
# Yang disimpan di sini adalah PENGATURAN agar backend memakai alur yang sama.
konfigurasi_rag = {
    'jenis': 'rag',
    'folder_db': 'data/vectordb',
    'nama_koleksi': koleksi.name,
    'nama_embedding': NAMA_EMBEDDING,
    'jarak': 'cosine',
    'k': 3,
    'batas_jarak': BATAS_JARAK,
    'base_url': str(client.base_url),
    'model_llm': MODEL_LLM,
    'temperature': 0,
    'max_tokens': 2000,
    'system_prompt': 'Jawab singkat dalam bahasa Indonesia.',
    'template_prompt': ('Jawab pertanyaan HANYA berdasarkan konteks berikut. '
                        'Jika tidak ada di konteks, tulis: {tidak_ada}\n\n'
                        'Konteks:\n{konteks}\n\n'
                        'Pertanyaan: {pertanyaan}\nJawaban singkat:'),
    'pesan_tidak_ada': 'Maaf, informasi tidak ditemukan di dokumen.',
    'jumlah_dokumen': koleksi.count()}

os.makedirs('models', exist_ok=True)
with open('models/konfigurasi_rag.json', 'w') as f:
    json.dump(konfigurasi_rag, f, indent=2, ensure_ascii=False)

# Cek: buka kembali konfigurasi lalu simulasikan tanya jawab
with open('models/konfigurasi_rag.json') as f:
    cfg = json.load(f)

def rag_simulasi(pertanyaan):
    """Alur R-A-G yang sepenuhnya mengikuti isi file konfigurasi."""
    h = koleksi.query(query_texts=[pertanyaan], n_results=cfg['k'])
    ids, teks, jarak = h['ids'][0], h['documents'][0], h['distances'][0]
    if jarak[0] > cfg['batas_jarak']:                          # R
        return ids, jarak, cfg['pesan_tidak_ada']
    konteks = '\n'.join(f'[{i}] {t}' for i, t in zip(ids, teks))
    prompt = cfg['template_prompt'].format(                     # A
        konteks=konteks, pertanyaan=pertanyaan,
        tidak_ada=cfg['pesan_tidak_ada'])
    respon = client.chat.completions.create(                    # G
        model=cfg['model_llm'],
        messages=[{'role': 'system', 'content': cfg['system_prompt']},
                  {'role': 'user', 'content': prompt}],
        temperature=cfg['temperature'], max_tokens=cfg['max_tokens'])
    return ids, jarak, respon.choices[0].message.content.strip()

print('Tersimpan : models/konfigurasi_rag.json')
print(f"Database  : {cfg['folder_db']} ({cfg['jumlah_dokumen']} dokumen), "
      f"batas jarak {cfg['batas_jarak']}")
print('=' * 70)

contoh_rag = {}
for no, q in enumerate(['Berapa persen bobot UTS?',
                        'Kapan ujian tengah semester diadakan?',
                        'Siapa nama asisten dosen kelas B?'], start=1):
    ids, jarak, jawaban = rag_simulasi(q)
    contoh_rag[q] = jawaban
    lolos = jarak[0] <= cfg['batas_jarak']
    print(f'PERTANYAAN {no}')
    print(f'  INPUT  : {q}')
    print('  PROSES : dokumen terdekat ' + ', '.join(
        f'{i} (jarak {j:.3f})' for i, j in zip(ids, jarak)))
    print(f"           jarak terdekat {jarak[0]:.3f} "
          f"{'<=' if lolos else '>'} batas {cfg['batas_jarak']} -> "
          f"{'dikirim ke LLM' if lolos else 'tidak dikirim ke LLM'}")
    print(f'  OUTPUT : {jawaban}')
    print('-' * 70)
    time.sleep(1)

**Penjelasan API:**

- `json.dump(isi, f, indent=2)`: menyimpan dictionary ke file JSON yang rapi. `json.load(f)` membukanya kembali.
- `koleksi.name` dan `koleksi.count()`: nama collection dan jumlah dokumen di vector database hasil langkah 2.4 Hari 2.
- `str(client.base_url)`: alamat server LLM yang dipakai (Groq).
- `template_prompt`: kalimat prompt dengan tempat kosong `{konteks}`, `{pertanyaan}`, dan `{tidak_ada}`. Method `.format(...)` mengisi tempat kosong tersebut.
- `rag_simulasi()`: alur R–A–G yang **hanya** memakai isi file konfigurasi. Jika fungsi ini berhasil, backend pun bisa menjalankan RAG dengan file yang sama.

> ✍️ **Amati:** pada baris PROSES terlihat dokumen yang ditemukan beserta jaraknya, lalu keputusan apakah pertanyaan dikirim ke LLM. Jika pertanyaan "ujian tengah semester" ditolak padahal dokumennya ada, nilai `BATAS_JARAK` terlalu ketat.

## Sel 2 – LLM Classification: Menyimpan Konfigurasi Prompt Terbaik
Dipilih satu konfigurasi final antara zero-shot dan few-shot berdasarkan akurasi pada evaluation set (langkah 2.8 Hari 2). Jika akurasinya sama, dipilih zero-shot karena prompt-nya lebih pendek sehingga lebih hemat token. Setelah disimpan, konfigurasi dibuka kembali untuk mengklasifikasikan 3 kode **baru** yang tidak ada di evaluation set.

In [ ]:
# SEL 2 - LLM CLASSIFICATION: simpan konfigurasi prompt terbaik
import os
import json
import time

# Pilih 1 konfigurasi final: akurasi tertinggi (langkah 2.8 Hari 2).
# Jika sama, pilih zero-shot karena prompt lebih pendek (lebih hemat token).
akurasi_zero = (data_error['zero-shot'] == data_error['label']).mean()
akurasi_few = (data_error['few-shot'] == data_error['label']).mean()
mode = 'few-shot' if akurasi_few > akurasi_zero else 'zero-shot'

konfigurasi_klasifikasi = {
    'jenis': 'llm_classification',
    'mode': mode,
    'kategori': JENIS,
    'instruksi': instruksi,
    'contoh': contoh if mode == 'few-shot' else '',
    'template_prompt': '{instruksi}\n\n{contoh}Kode:\n{kode}\nJawaban:',
    'jika_tidak_cocok': 'Lainnya',
    'base_url': str(client.base_url),
    'model_llm': MODEL_LLM,
    'temperature': 0,
    'max_tokens': 2000,
    'akurasi_uji': round(max(akurasi_zero, akurasi_few), 3)}

os.makedirs('models', exist_ok=True)
with open('models/konfigurasi_klasifikasi.json', 'w') as f:
    json.dump(konfigurasi_klasifikasi, f, indent=2, ensure_ascii=False)

# Cek: buka kembali konfigurasi lalu klasifikasikan 3 kode BARU
with open('models/konfigurasi_klasifikasi.json') as f:
    cfg_k = json.load(f)

def klasifikasi_simulasi(kode):
    """Menyusun prompt dari konfigurasi, lalu mengambil kategori jawaban."""
    prompt = cfg_k['template_prompt'].format(
        instruksi=cfg_k['instruksi'], contoh=cfg_k['contoh'], kode=kode)
    respon = client.chat.completions.create(
        model=cfg_k['model_llm'],
        messages=[{'role': 'user', 'content': prompt}],
        temperature=cfg_k['temperature'], max_tokens=cfg_k['max_tokens'])
    teks = respon.choices[0].message.content.strip()
    kategori = next((k for k in cfg_k['kategori']
                     if k.lower() in teks.lower()), cfg_k['jika_tidak_cocok'])
    return teks, kategori

print('Tersimpan : models/konfigurasi_klasifikasi.json')
print(f"Mode      : {cfg_k['mode']} (akurasi zero-shot {akurasi_zero:.0%}, "
      f"few-shot {akurasi_few:.0%})")
print('=' * 70)

contoh_klasifikasi = {}
kode_baru = ["nama = 'Budi'\nprint(nam)",
             "harga = [5000, 7000]\nprint(harga[2])",
             "total = '10' + 5"]
for no, kode in enumerate(kode_baru, start=1):
    teks, kategori = klasifikasi_simulasi(kode)
    contoh_klasifikasi[kode] = kategori
    print(f'KODE {no}')
    print('  INPUT  : ' + kode.replace('\n', '\n           '))
    print(f"  PROSES : prompt {cfg_k['mode']} dikirim ke LLM, "
          f'jawaban mentah: {teks!r}')
    print(f'  OUTPUT : {kategori}')
    print('-' * 70)
    time.sleep(1)

**Penjelasan API:**

- `(data_error['zero-shot'] == data_error['label']).mean()`: akurasi, yaitu proporsi tebakan yang sama dengan kunci jawaban.
- `contoh`: contoh few-shot hanya disimpan jika mode final adalah few-shot. Untuk zero-shot diisi teks kosong.
- `template_prompt`: `{instruksi}`, `{contoh}`, dan `{kode}` diisi dengan `.format(...)`, sama seperti fungsi `buat_prompt()` Hari 2.
- `jika_tidak_cocok`: kategori cadangan (`'Lainnya'`) jika jawaban LLM tidak memuat satu pun kategori.
- `{teks!r}`: menampilkan jawaban mentah LLM apa adanya, sehingga terlihat bagaimana kategori diambil dari jawaban.

## Sel 3 – Membuat requirements.txt dan .env.example
`requirements.txt` berisi library beserta versinya. `.env.example` berisi contoh isi file `.env` **tanpa API key asli**. Tim backend menyalinnya menjadi `.env` lalu mengisi API key milik mereka sendiri.

In [ ]:
# SEL 3 - requirements.txt dan .env.example
import sys
from importlib.metadata import version

paket = ['openai', 'chromadb', 'sentence-transformers', 'python-dotenv']
isi_req = ''
for p in paket:
    try:
        isi_req += f'{p}=={version(p)}\n'
    except Exception:                  # library belum terpasang
        isi_req += f'{p}\n'
with open('requirements.txt', 'w') as f:
    f.write(isi_req)

# .env.example: contoh isi .env TANPA API key asli
with open('.env.example', 'w') as f:
    f.write('GROQ_API_KEY=isi_dengan_api_key_groq\n')

print('Python', sys.version.split()[0])
print(isi_req)
print('.env.example dibuat (API key asli TIDAK ikut diserahkan)')

**Penjelasan API:**

- `version('nama-library')`: mengambil versi library yang terpasang. Jika library belum terpasang, namanya tetap ditulis tanpa versi.
- API key **tidak pernah** ditulis ke file yang diserahkan. API key hanya disimpan di server backend (file `.env`), tidak di aplikasi web atau mobile.

## Sel 4 – Membuat Dokumen Kontrak (KONTRAK.md)
Dokumen kontrak menjelaskan kepada tim backend: file yang dipakai, **urutan langkah** yang harus dijalankan, pengaturannya, dan contoh input–output. Contoh input–output diambil dari hasil simulasi Sel 1 dan Sel 2, sehingga **tidak ada request tambahan ke LLM**. Sel ini hanya menulis bagian untuk konfigurasi yang file-nya ada.

In [ ]:
# SEL 4 - KONTRAK.md: cara memakai, konfigurasi, contoh input dan output
import os
import sys
import json

def blok_json(isi):
    teks = json.dumps(isi, indent=2, ensure_ascii=False)
    return '```json\n' + teks + '\n```'

bagian = [f"""# Kontrak Komponen AI (LLM)

Dibuat dengan Python {sys.version.split()[0]}. Pasang library dengan:
`pip install -r requirements.txt`. Salin `.env.example` menjadi `.env`,
lalu isi GROQ_API_KEY milik tim backend (API key tidak boleh ada di
aplikasi web/mobile).
"""]

# ---------- Kontrak RAG ----------
if os.path.exists('models/konfigurasi_rag.json'):
    with open('models/konfigurasi_rag.json') as f:
        c = json.load(f)
    daftar = list(contoh_rag.items()) if 'contoh_rag' in globals() else []
    contoh_io = '\n'.join(
        f"Input:\n{blok_json({'pertanyaan': q})}\nOutput:\n"
        f"{blok_json({'jawaban': j})}" for q, j in daftar)
    bagian.append(f"""## 1. RAG: Tanya Jawab Berdasarkan Dokumen
File: folder `{c['folder_db']}` (vector database ChromaDB, collection
`{c['nama_koleksi']}`, {c['jumlah_dokumen']} dokumen) dan
`models/konfigurasi_rag.json` (seluruh pengaturan).

### Alur yang harus dijalankan backend untuk setiap pertanyaan
1. **Retrieve:** cari {c['k']} dokumen terdekat di collection memakai
   embedding `{c['nama_embedding']}` (jarak {c['jarak']}).
2. Jika jarak dokumen terdekat > {c['batas_jarak']}: langsung jawab
   "{c['pesan_tidak_ada']}" tanpa memanggil LLM.
3. **Augment:** isi `template_prompt` dengan konteks (format `[id] teks`,
   satu dokumen per baris), pertanyaan, dan pesan tidak ada.
4. **Generate:** kirim ke `{c['model_llm']}` di `{c['base_url']}` dengan
   system prompt, temperature {c['temperature']}, max_tokens
   {c['max_tokens']}.

### Contoh input dan output
{contoh_io}
""")

# ---------- Kontrak LLM classification ----------
if os.path.exists('models/konfigurasi_klasifikasi.json'):
    with open('models/konfigurasi_klasifikasi.json') as f:
        c = json.load(f)
    daftar = (list(contoh_klasifikasi.items())
              if 'contoh_klasifikasi' in globals() else [])
    contoh_io = '\n'.join(
        f"Input:\n{blok_json({'kode': k})}\nOutput:\n"
        f"{blok_json({'kategori': v})}" for k, v in daftar)
    bagian.append(f"""## 2. LLM Classification: Jenis Error Kode Python
File: `models/konfigurasi_klasifikasi.json`. Mode final: {c['mode']}
(akurasi evaluation set {c['akurasi_uji']:.0%}).

### Alur yang harus dijalankan backend untuk setiap input
1. Isi `template_prompt` dengan `instruksi`, `contoh`, dan kode dari
   pengguna.
2. Kirim ke `{c['model_llm']}` di `{c['base_url']}` (temperature
   {c['temperature']}, max_tokens {c['max_tokens']}).
3. Ambil kategori pertama dari daftar `kategori` yang muncul di jawaban
   LLM (tanpa membedakan huruf besar/kecil). Jika tidak ada, hasilnya
   "{c['jika_tidak_cocok']}".

### Kategori yang mungkin
{', '.join(c['kategori'])}

### Contoh input dan output
{contoh_io}
""")

kontrak = '\n'.join(bagian)
with open('KONTRAK.md', 'w') as f:
    f.write(kontrak)
print(kontrak)

**Penjelasan API:**

- `os.path.exists(path)`: memeriksa apakah file konfigurasi ada. Tim RAG saja akan mendapat kontrak bagian RAG saja.
- `'contoh_rag' in globals()`: memeriksa apakah hasil simulasi Sel 1 masih ada di memori.
- Bagian **"Alur yang harus dijalankan backend"** adalah inti kontrak: backend harus menjalankan langkah yang sama persis agar jawaban sistem sama dengan hasil evaluasi.

> ✍️ **Periksa:** baca isi KONTRAK.md yang tercetak. Apakah tim backend bisa menjalankan RAG atau klasifikasi hanya dengan membaca dokumen ini dan file konfigurasi?

## Sel 5 – Menyatukan dan Mengunduh Hasil
Folder vector database, file konfigurasi (.json), `requirements.txt`, `.env.example`, dan `KONTRAK.md` disatukan dalam `hasil_llm.zip`, lalu diunduh. File zip inilah yang diserahkan kepada tim backend atau di-commit ke repository tim.

In [ ]:
# SEL 5 - Satukan vector database, konfigurasi, dan dokumen, lalu unduh
from google.colab import files

!zip -q -r hasil_llm.zip data/vectordb models/*.json requirements.txt .env.example KONTRAK.md
!unzip -l hasil_llm.zip
files.download('hasil_llm.zip')

**Penjelasan API:**

- `!zip -q -r ...`: menyatukan folder dan file menjadi satu zip. Opsi `-r` menyertakan seluruh isi folder `data/vectordb`.
- `!unzip -l`: menampilkan daftar isi zip untuk memastikan semua file sudah masuk.
- `files.download(path)`: mengunduh file dari Colab ke komputer.